In [1]:
# Импортируем необходимые библиотеки
import torch # Основная библиотека для работы с тензорами и нейронными сетями
import torch.nn as nn # Модуль для построения нейронных сетей
from torchvision.datasets import MNIST # Датасет MNIST для рукописных цифр
import torchvision.transforms as T # Модуль для преобразований изображений

from torch.utils.data import DataLoader # Для загрузки данных батчами
from IPython.display import clear_output # Для очистки вывода в Jupyter/Colab
from tqdm import tqdm # Для отображения прогресса выполнения циклов
import matplotlib.pyplot as plt # Для построения графиков
from torch.optim import Adam # Оптимизатор Adam
import numpy as np # Для работы с массивами (в частности, для случайных выборок)
from PIL import Image # Для работы с изображениями (Pillow)
import math # Для математических операций (например, sqrt)
from torchvision.utils import make_grid # Для создания сетки из изображений
import gc

# Полностью очищаем оперативную и видеопамять от старых тензоров
gc.collect()
torch.cuda.empty_cache()
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.backends.cudnn.benchmark = True

In [2]:
# Определяем преобразования для изображений:
# 1. Изменяем размер изображения до 64x64 пикселей. Это увеличит детализацию и сделает
#    задачу для автоэнкодера более сложной, чем стандартные 28x28.
# 2. Преобразуем изображение из PIL Image в PyTorch Tensor.
mnist_transforms = T.Compose(
    [
        T.Resize((64, 64)),
        T.ToTensor(), # Преобразует изображение в тензор и нормализует пиксели до диапазона [0, 1]
    ]
)

# Загружаем обучающий датасет MNIST. Если его нет, он будет загружен автоматически.
train_dataset = MNIST('mnist', train=True, transform=mnist_transforms, download=True)
# Загружаем тестовый (валидационный) датасет MNIST.
valid_dataset = MNIST('mnist', train=False, transform=mnist_transforms, download=True)

# Создаем DataLoader'ы для удобной загрузки данных батчами:
# batch_size=64: Количество изображений в каждом батче.
# shuffle=True: Перемешивает данные в каждой эпохе для обучающего датасета,
#               чтобы модель не запоминала порядок.
# num_workers=8: Количество подпроцессов для загрузки данных (ускоряет).
# pin_memory=True: Загружает данные в "закрепленную" память, что может ускорить
#                  передачу данных на GPU.
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=8, pin_memory=True)
valid_loader = DataLoader(valid_dataset, batch_size=64, shuffle=False, num_workers=8, pin_memory=True)

100%|██████████| 9.91M/9.91M [00:00<00:00, 47.5MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.67MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 10.2MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.50MB/s]
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [3]:
def train(model):
    model.train() # Переводим модель в режим обучения (активирует dropout, batchnorm и т.д.)

    train_loss = 0 # Переменная для накопления общей ошибки за эпоху

    # Итерируемся по обучающему DataLoader'у. tqdm добавляет красивую полосу прогресса.
    # В автоэнкодере нам не нужны метки (y), поэтому мы используем _ для их игнорирования.
    for x, _ in tqdm(train_loader, desc='Train'):
        x = x.to(device) # Перемещаем входные изображения на используемое устройство (GPU/CPU)

        optimizer.zero_grad() # Обнуляем градиенты перед началом нового шага оптимизации

        output = model(x) # Пропускаем изображение через модель (прямой проход)
        # Для автоэнкодера выход - это восстановленное изображение,
        # а целью является само входное изображение (x).
        loss = loss_fn(output, x) # Вычисляем ошибку между выходом модели и исходным изображением

        train_loss += loss.item() # Добавляем значение ошибки текущего батча к общей ошибке

        loss.backward() # Вычисляем градиенты для всех обучаемых параметров модели

        optimizer.step() # Обновляем веса модели с помощью оптимизатора

    train_loss /= len(train_loader) # Усредняем ошибку по всем батчам
    return train_loss

In [4]:
@torch.inference_mode() # Декоратор для отключения вычисления градиентов,
                        # что экономит память и ускоряет вычисления во время оценки.
def evaluate(model, loader):
    model.eval() # Переводим модель в режим оценки (отключает dropout, замораживает batchnorm и т.д.)

    total_loss = 0 # Переменная для накопления общей ошибки

    for x, _ in tqdm(loader, desc='Evaluation'):
        x = x.to(device) # Перемещаем изображения на устройство

        output = model(x) # Прямой проход

        loss = loss_fn(output, x) # Вычисление ошибки

        total_loss += loss.item() # Накопление ошибки

    total_loss /= len(loader) # Усреднение ошибки
    return total_loss

In [5]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

print("Используемое устройство:", device)
if torch.cuda.is_available():
    print("Имя GPU:", torch.cuda.get_device_name())

# Функция потерь: Mean Squared Error (MSE).
# Она вычисляет средний квадрат разности между пикселями исходного и восстановленного изображений.
# MSE хорошо подходит для задач реконструкции, так как штрафует за большие ошибки и поощряет точное восстановление.
loss_fn = nn.MSELoss()

Используемое устройство: cuda:0
Имя GPU: Tesla T4


In [6]:
@torch.inference_mode()
def get_latent_labels(model, loader, count: int=None):
    """
    Генерирует латентные представления (эмбеддинги) и соответствующие метки
    для определенного количества изображений из DataLoader'а.
    """
    model.eval() # Переводим модель в режим оценки
    
    total = 0 # Счетчик обработанных изображений
    
    latent = [] # Список для хранения эмбеддингов
    labels = [] # Список для хранения меток
    
    for x, y in loader:
        # Кодируем изображения в латентное пространство, перемещаем на CPU.
        z = model.encode(x.to(device)).cpu()
        
        latent.append(z) # Добавляем эмбеддинги
        labels.append(y) # Добавляем метки
        
        total += y.shape[0] # Увеличиваем счетчик
        
        # Если достигнуто требуемое количество, прерываем цикл.
        if count is not None and total >= count:
            break
            
    # Объединяем все эмбеддинги и метки в один тензор и преобразуем в NumPy массивы.
    latent = torch.cat(latent)[:count].numpy()
    labels = torch.cat(labels)[:count].numpy()
    
    return latent, labels

In [7]:
def save_embeddings(x_train, y_train, x_valid, y_valid):
    x_train_tensor = torch.from_numpy(x_train)
    y_train_tensor = torch.from_numpy(y_train)
    x_valid_tensor = torch.from_numpy(x_valid)
    y_valid_tensor = torch.from_numpy(y_valid)
    assert x_train_tensor.shape[0] == 1000
    assert x_valid_tensor.shape[0] == 10000
    
    assert y_train_tensor.shape[0] == 1000
    assert y_valid_tensor.shape[0] == 10000

    torch.save(
        {
            'x_train': x_train_tensor,
            'y_train': y_train_tensor,
            'x_valid': x_valid_tensor,
            'y_valid': y_valid_tensor
        },
        'embeddings.pt'
    )

In [8]:
from sklearn.ensemble import RandomForestClassifier
def whole_train_valid_cycle(model, num_epochs, title):

    for epoch in range(num_epochs):
        print(f"Epoch {epoch + 1}/{num_epochs}")
        train_loss = train(model) 
        valid_loss = evaluate(model, valid_loader)
        print(f"Train Loss: {train_loss:.6f}, Valid Loss: {valid_loss:.6f}")
        
        x, y = get_latent_labels(model, train_loader, 1000)
        clf = RandomForestClassifier(random_state=0).fit(x, y)
        x_valid, y_valid = get_latent_labels(model, valid_loader)
        valid_accuracy = (clf.predict(x_valid) == y_valid).mean()
        print(f"valid_accuracy={valid_accuracy}")
        if valid_accuracy >= 0.90:
            print("Целевая метрика в 90% достигнута!")

            # Сохраняем строго через torch.save на диск в текущую папку
            # Переносим на CPU для гарантированной совместимости с чекером
            save_embeddings(x, y, x_valid, y_valid)
            print("Тензор предсказаний успешно записан в файл 'predictions.pt'")
            break
        

In [9]:
class Block(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int, stride: int = 1, upsample: bool = False):
        super().__init__()
        self.upsample = upsample # Флаг, указывающий, нужно ли делать интерполяцию (для декодера)

        # Сверточный слой
        # in_channels: количество входных каналов (например, 1 для ч/б MNIST)
        # out_channels: количество выходных каналов (глубина признаковых карт)
        # kernel_size: размер ядра свертки (3x3 в данном случае)
        # stride: шаг свертки. stride=2 уменьшает размер изображения в 2 раза
        # padding=1: добавляет 1 пиксель отступов по краям, чтобы размер выходной карты
        #            сохранялся при stride=1, или уменьшался ровно в 2 раза при stride=2.
        # bias=False: отключаем смещение, так как Batch Normalization все равно введет свое смещение.
        self.conv = nn.Conv2d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=1,
            bias=False
        )
        self.norm = nn.BatchNorm2d(out_channels) # Batch Normalization для стабилизации обучения
        self.act = nn.LeakyReLU(0.2) # Leaky ReLU активация: предотвращает "умирание" нейронов,
                                     # допуская небольшой отрицательный градиент для отрицательных входов.

    def forward(self, x):
        # Если блок используется в декодере и установлен флаг upsample,
        # то увеличиваем размер изображения в 2 раза перед сверткой.
        # mode='bilinear': используется билинейная интерполяция для гладкого увеличения.
        if self.upsample:
            x = nn.functional.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False, recompute_scale_factor=False)

        # Выполняем свертку, нормализацию и активацию.
        return self.act(self.norm(self.conv(x)))

In [10]:
class AutoEncoder(nn.Module):
    def __init__(self, in_channels: int, base_block: nn.Module = Block, base_size: int = 32, num_blocks: int = 4, latent_dim: int = 128):
        super().__init__()

        self.base_size = base_size
        self.num_blocks = num_blocks
        self.latent_dim = latent_dim
        
        # Вычисляем пространственный размер карты после сверток (для 64x64 при 4 блоках это 4x4)
        self.conv_feat_size = 64 // (2 ** num_blocks) 
        self.flatten_dim = base_size * (self.conv_feat_size ** 2) # 32 * 4 * 4 = 512

        # --- Создание Энкодера ---
        encoder_blocks = []
        for i in range(num_blocks):
            encoder_blocks.append(
                base_block(
                    in_channels=base_size if i else in_channels,
                    out_channels=base_size,
                    kernel_size=3,
                    stride=2
                )
            )
        encoder_blocks.append(
            base_block(in_channels=base_size, out_channels=base_size, kernel_size=3).conv
        )
        self.encoder_conv = nn.Sequential(*encoder_blocks)
        
        # Линейный слой сжатия до нужного N <= 128
        self.fc_encode = nn.Linear(self.flatten_dim, latent_dim)

        # --- Создание Декодера ---
        # Линейный слой восстановления размерности из вектора 128 обратно в 512 для сверток
        self.fc_decode = nn.Linear(latent_dim, self.flatten_dim)
        
        decoder_blocks = []
        for i in range(num_blocks):
            decoder_blocks.append(
                base_block(in_channels=base_size, out_channels=base_size, kernel_size=3, upsample=True)
            )
        decoder_blocks.append(
            base_block(in_channels=base_size, out_channels=in_channels, kernel_size=3).conv
        )
        decoder_blocks.append(nn.Sigmoid())

        self.decoder_conv = nn.Sequential(*decoder_blocks)
        self.flatten = nn.Flatten()

    def forward(self, x):
        # Полный проход
        latent = self.encode(x)
        return self.decode(latent)

    def encode(self, x):
        # Извлекаем признаки свертками -> сглаживаем -> сжимаем до 128
        features = self.flatten(self.encoder_conv(x))
        return self.fc_encode(features)

    def decode(self, z):
        # Восстанавливаем из 128 в 512 -> меняем форму в 3D тензор -> разворачиваем свертками
        features = self.fc_decode(z)
        features_3d = features.view(-1, self.base_size, self.conv_feat_size, self.conv_feat_size)
        return self.decoder_conv(features_3d)


In [11]:
# Инициализируем модель AutoEncoder:
# in_channels=1, так как изображения MNIST черно-белые (один канал).
model = AutoEncoder(in_channels=1).to(device) # Перемещаем модель на GPU

# Инициализируем оптимизатор Adam:
# Adam - это адаптивный оптимизатор, который хорошо работает в большинстве случаев.
# model.parameters(): передаем все обучаемые параметры модели.
# lr=1e-3: начальная скорость обучения.
optimizer = Adam(model.parameters(), lr=1e-3)

In [12]:
whole_train_valid_cycle(model, 30, 'AutoEncoder MNIST')

Epoch 1/30


Train:   0%|          | 0/938 [00:00<?, ?it/s]/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Evaluation: 100%|██████████| 157/157 [00:01<00:00, 104.89it/s]

Train Loss: 0.010125, Valid Loss: 0.004259



/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


valid_accuracy=0.934
Целевая метрика в 90% достигнута!
Тензор предсказаний успешно записан в файл 'predictions.pt'
